# COMPARISON — Temporal-First Graph Network (TFGN) ablation ladder

Implements `DOCS/temporal-first-ablation.md`'s 2026-08-24 "Evaluation & Comparison
Protocol" addendum over the ladder in `DOCS/flipped/PLAN.md` Phase 4 /
`CLASSIFIER/experiments/temporal_first.yaml`. Every section through Tier 3 reads only
`run_summary["oof"]` / `oof_predictions.csv` — **never** `test_*` / `ext_*` keys. The
one frozen in-domain + one frozen OASIS-3 read live in the final section only, gated
behind an explicit flag (`RUN_FROZEN_READ`), so re-running this notebook does not
accidentally spend the ladder's one test read before it is meant to.

`source_experiment`-style: no training here — every number comes from
`outputs/<rung>-seed{42..45}/latest/run_summary.json` (`adapters.explain.resolve_source_run`),
written by `LONGITUDINAL_COMMON_DELCODE.ipynb`'s runs.

In [ ]:
# === Papermill parameters ===
EXPERIMENT_ID = None
MODE = None
MODEL = None
SEED = 42
WANDB_ENABLED = False
OUTPUT_DIR = None
RUN_DIR = None
RUN_NAME = None
# Tier-4 is gated: only flip this (and set FROZEN_WINNER_ID) once the FULL chain
# (S1 -> S1c_recon_random -> S2 -> S3 -> S4 -> S5 -> SENS) has reported against
# Tier 2/Tier 3 -- not merely once S1c-random has (DOCS/temporal-first-ablation.md
# Tier-4 gate, restated 2026-08-24).
RUN_FROZEN_READ = False
FROZEN_WINNER_ID = None       # the frozen S1-lineage winner's id prefix (no seed suffix),
                               # e.g. 'tfgn-s5-dualscore-pooled' once SENS has reported.
SECONDARY_SENSITIVITY_ID = None  # optional: a single id prefix, OR a list of id prefixes,
                                  # e.g. ['tfgn-s1b-ssl-pooled', 'tfgn-s5-dualscore-pooled'] --
                                  # secondary reads taken in this SAME pass and reported side
                                  # by side, never substituted for the primary. S5's read is
                                  # the interpretability layer's number (kept regardless of
                                  # AUC, PLAN.md section A), not a competing endpoint -- its
                                  # label makes that explicit below.


## Pipeline overview

Resolve each rung's 4 seed runs -> Tier 1 floors -> Tier 2 rung table + stopping rule (paired seed-level OOF ΔAUC) -> Tier 3 vetoes -> Tier 4 frozen reads (gated).

In [ ]:
import sys
from pathlib import Path
repo_root = Path('/mnt/e/fyassine/ad-early-detection')
model_root = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(model_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
    sys.path.insert(0, str(model_root))


In [ ]:
# reproducibility seeding -- must run before datasets / models.
from SHARED.seeding import set_seed, make_rng, make_torch_generator
set_seed(SEED)
rng = make_rng(SEED)
torch_gen = make_torch_generator(SEED)


In [ ]:
import json, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from adapters.explain import resolve_source_run
from common.comparison import paired_bootstrap_ci

warnings.filterwarnings('ignore')
print('Imports OK')


## Configuration — rung registry

One entry per ladder rung; extend this dict as S1c-SENS land (`DOCS/temporal-first-ablation.md` "The arms") -- nothing else in this notebook needs to change. `S0_demo` is the new Tier-1 demographics floor (`tfgn-s0-demo-pooled`); it has no runs yet until dispatched.

In [ ]:
SEEDS = [42, 43, 44, 45]

# name -> registry id prefix (seed appended as '-seed{42,43,44,45}').
RUNG_PREFIXES = {
    'S0a_logreg_drift':   'tfgn-s0-logreg-drift-pooled',
    'S0_demo':            'tfgn-s0-demo-pooled',
    'S0b_gelstm_frozen':  'tfgn-s0-gelstm-frozen-pooled',
    'S0c_gelstm_random':  'tfgn-s0-gelstm-random-pooled',
    'S0d_braintokengt':   'tfgn-s0-braintokengt-pooled',
    'S1_flip':            'tfgn-s1-flip-pooled',
    'S1b_ssl':             'tfgn-s1b-ssl-pooled',
    'S1c_recon_invalid':   'tfgn-s1c-recon-pooled',           # SUPERSEDED 2026-08-24 -- built on the
                                                                # since-reversed pretrained_finetuned fork;
                                                                # recorded as undecidable, kept ONLY out of
                                                                # RUNG_CHAIN/HEADLINE_CONTRASTS below (see the
                                                                # markdown cell after this one).
    'S1c_recon_random':    'tfgn-s1c-recon-random-pooled',    # protocol-valid re-run, node_lstm_init=random
    # S2-SENS all branch from S1 (not from S1c_recon_random, which the stopping rule
    # rejected) -- their Tier-2 comparison is against S1_flip directly, computed in
    # the section-F/G-style ad-hoc cells in DOCS/flipped/PLAN.md, not via RUNG_CHAIN
    # (which models a sequential chain and would misattribute these deltas against
    # S1c_recon_random if simply appended). Left out of RUNG_CHAIN/HEADLINE_CONTRASTS
    # deliberately; their OOF rows still surface in RUNG_SUMMARY_TABLE below.
    'S2_gate':             'tfgn-s2-gate-pooled',
    'S3_fusion':           'tfgn-s3-fusion-pooled',           # VOID -- inert knob, bit-identical to S1 (PLAN.md section B)
    'S4_attnpool':         'tfgn-s4-attnpool-pooled',
    'S5_dualscore':        'tfgn-s5-dualscore-pooled',        # kept regardless of AUC -- interpretability layer (PLAN.md section A)
    'SENS':                'tfgn-sens-minvisits3-pooled',     # min_visits=3, N=140 -- not fold-matched vs S1 (PLAN.md section J)
    # Matched-window SOTA comparison (DOCS/flipped/PLAN.md addendum) -- Table A only,
    # never Tier 1-3 gated against the ladder (PLAN.md "Not a ladder rung"). S0d_braintokengt
    # above is Table A's fourth row, reused verbatim, not re-run.
    'W3_gelstm_frozen':    'tfgn-w3-gelstm-frozen-pooled',
    'W3_gelstm_random':    'tfgn-w3-gelstm-random-pooled',
    'W3_tfgn_winner':      'tfgn-w3-winner-pooled',
}

# The pre-registered chain (DOCS/temporal-first-ablation.md "The arms"), CORRECTED
# 2026-08-24 per the "S1b fork decision" correcting addendum: S1b is dropped by
# Tier 2's one-SE tie-breaker (see the tie-breaker cell below) and is therefore NOT
# in the primary chain -- it is scored separately as a sensitivity arm. S1c_recon
# (the original, pretrained_finetuned run) is likewise excluded from the chain --
# it tests an unregistered configuration, not the S1c question. The chain's Tier-2
# stopping-rule comparison is each rung against the one immediately before it here.
RUNG_CHAIN = ['S0c_gelstm_random', 'S1_flip', 'S1c_recon_random']  # extend: ..., 'S2_gate', ...

# Sensitivity arms: reported (fold-matched AND pooled OOF Δ vs their reference rung)
# but never part of RUNG_CHAIN and never selected on. S1b's own OOF rows still
# appear in RUNG_SUMMARY_TABLE via RUNG_PREFIXES above.
SENSITIVITY_CONTRASTS = {
    'S1b_vs_S1': ('S1_flip', 'S1b_ssl'),  # the corrected fork read -- see the tie-breaker cell
}

HEADLINE_CONTRASTS = {
    'S0c_vs_S1': ('S0c_gelstm_random', 'S1_flip'),
    'S0b_vs_S1c': ('S0b_gelstm_frozen', 'S1c_recon_random'),  # the protocol-valid re-run, not S1c_recon_invalid
}


**S1c status note (2026-08-24).** `tfgn-s1c-recon-pooled` (`S1c_recon_invalid` above)
was built with `node_lstm_init: pretrained_finetuned`, inheriting the S1b fork decision
as it stood before the correcting addendum below reversed it. That run tests an
unregistered configuration and is recorded as **undecidable**, not as a result for or
against the flip -- see `DOCS/temporal-first-ablation.md` "S1c (2026-08-24 run) --
recorded as undecidable". It is intentionally excluded from `RUNG_CHAIN` and from
`HEADLINE_CONTRASTS`; only `S1c_recon_random` (`node_lstm_init: random`, the
protocol-valid re-run) participates in either. Its OOF rows remain visible in
`RUNG_SUMMARY_TABLE` for the record, labelled accordingly.

In [ ]:
def resolve_rung_runs(prefix):
    """exp_id -> run_dir for every seed of one rung; missing runs are skipped, not fatal
    (this notebook must stay runnable before every rung/seed has been dispatched)."""
    run_dirs = {}
    for seed in SEEDS:
        exp_id = f'{prefix}-seed{seed}'
        try:
            run_dirs[exp_id] = resolve_source_run(exp_id, classifier_root=model_root)
        except FileNotFoundError:
            print(f'  [skip] {exp_id}: no run yet')
    return run_dirs


RUNG_RUN_DIRS = {name: resolve_rung_runs(prefix) for name, prefix in RUNG_PREFIXES.items()}
for name, dirs in RUNG_RUN_DIRS.items():
    print(f'{name:22s} {len(dirs)}/{len(SEEDS)} seeds resolved')


In [ ]:
def load_summary(run_dir):
    """None (not a raised error) if the run hasn't written run_summary.json yet --
    a run still 'running' on the other host is a normal state this notebook must
    tolerate, not treat as missing/broken."""
    path = run_dir / 'run_summary.json'
    return json.loads(path.read_text()) if path.is_file() else None


def load_calibration(run_dir):
    path = run_dir / 'calibration.json'
    return json.loads(path.read_text()) if path.is_file() else {}


def rung_oof_rows(run_dirs):
    """One row per seed with that seed's oof.* metrics (empty 'oof' -> pre-addendum,
    not-yet-re-run artifact -- flagged, not silently dropped)."""
    rows = []
    for exp_id, run_dir in run_dirs.items():
        summary = load_summary(run_dir)
        if summary is None:
            print(f'  [in-flight] {exp_id}: no run_summary.json yet (run still running?) -- skipped.')
            continue
        oof = summary.get('oof')
        if not oof:
            print(f'  [stale] {exp_id}: no run_summary["oof"] -- re-run under the '
                  '2026-08-24 addendum contract before trusting this rung\'s table row.')
            continue
        cal = load_calibration(run_dir)
        row = {'exp_id': exp_id, 'seed': int(exp_id.rsplit('seed', 1)[-1]),
               'cohort_probe_auc': summary.get('cohort_probe_auc'),
               'ece_oof_cal': cal.get('ece_oof_cal')}
        row.update(oof)
        rows.append(row)
    return pd.DataFrame(rows)


## Tier 1 — floor gates

In [ ]:
# Demographics floor: tfgn-s0-demo-pooled (feature_set='demo', [age, sex] only).
DEMO_FLOOR = rung_oof_rows(RUNG_RUN_DIRS.get('S0_demo', {}))
if not DEMO_FLOOR.empty:
    print('Demographics floor (age+sex only), OOF AUC:',
          f"{DEMO_FLOOR['oof_auc'].mean():.4f} +/- {DEMO_FLOOR['oof_auc'].std():.4f}")
else:
    print('Demographics floor: no runs yet (dispatch tfgn-s0-demo-pooled-seed{42..45}).')

# SSL persistence baseline -- already computed by LONGITUDINAL_TFGN_SSL_POOLED.ipynb
# itself; nothing to compute here, just surface it.
_p2_dir = resolve_source_run('tfgn-nodelstm-ssl-pooled', classifier_root=model_root)
_p2_summary = load_summary(_p2_dir) or {}
PERSISTENCE_BASELINE = _p2_summary.get('persistence_baseline', {})
print('P2 SSL persistence baseline:', PERSISTENCE_BASELINE)


## Tier 2 — rung table (OOF only) + stopping rule

In [ ]:
RUNG_TABLES = {name: rung_oof_rows(dirs) for name, dirs in RUNG_RUN_DIRS.items()}

# Batch-5 verdicts (DOCS/temporal-first-ablation.md "Batch 5 verdicts" addendum,
# DOCS/flipped/PLAN.md sections A/B/E/J) -- stated in the table itself, not only
# in prose, so a reader scanning RUNG_SUMMARY_TABLE alone cannot miss that S3's
# row is not a result.
RUNG_STATUS = {
    'S1_flip':            'PRIMARY -- selected by the stopping rule',
    'S1b_ssl':             'sensitivity arm -- one-SE tie-breaker selects S1 over this (Tier-2 cell above)',
    'S1c_recon_invalid':   'UNDECIDABLE -- built on the since-reversed pretrained_finetuned fork, not a result',
    'S1c_recon_random':    'dropped -- fold-matched Delta vs S1 = -0.1587 +/- 0.0099',
    'S2_gate':             'dropped -- fold-matched Delta vs S1 = -0.0064 +/- 0.0075',
    'S3_fusion':           'VOID -- fusion knob is inert under recon_target=none (models.py:175-183); '
                            'oof_predictions.csv is bit-identical to S1 on all 4 seeds, not a result',
    'S4_attnpool':         'dropped -- fold-matched Delta vs S1 = -0.0558 +/- 0.0058',
    'S5_dualscore':        'KEPT -- interpretability layer, classification-neutral (|Delta|<SE vs S1), '
                            'kept regardless of AUC per its pre-registration (PLAN.md section A)',
    'SENS':                'not fold-matched vs S1 (different N/subjects) -- see the SENS-restricted '
                            'comparison cell below for the like-for-like read (PLAN.md section E)',
}

_cols = ['oof_auc', 'oof_pr_auc', 'oof_balanced_accuracy', 'oof_static_n1_auc', 'cohort_probe_auc']
summary_rows = []
for name, df in RUNG_TABLES.items():
    if df.empty:
        summary_rows.append({'rung': name, 'n_seeds': 0, 'status': RUNG_STATUS.get(name, '')})
        continue
    row = {'rung': name, 'n_seeds': len(df), 'status': RUNG_STATUS.get(name, '')}
    for c in _cols:
        if c in df.columns:
            row[f'{c}_mean'] = df[c].mean()
            row[f'{c}_sd'] = df[c].std()
    cohort_cols = [c for c in df.columns if c.startswith('oof_auc_') and c != 'oof_auc']
    for c in cohort_cols:
        row[f'{c}_mean'] = df[c].mean()
    summary_rows.append(row)

RUNG_SUMMARY_TABLE = pd.DataFrame(summary_rows).set_index('rung')
RUNG_SUMMARY_TABLE


In [ ]:
def per_fold_auc(run_dir):
    """fold -> OOF AUC, from oof_predictions.csv. The StratifiedGroupKFold split in
    common.crossval.run_kfold_cv takes no seed/shuffle, so fold i is the SAME subject
    group across every seed and every rung of one dataset -- this is what makes
    fold-matched pairing across arms/seeds valid."""
    path = run_dir / 'oof_predictions.csv'
    if not path.is_file():
        return {}
    df = pd.read_csv(path)
    out = {}
    for fold, sub in df.groupby('fold'):
        if sub['label'].nunique() > 1:
            out[int(fold)] = roc_auc_score(sub['label'], sub['prob'])
    return out


def stopping_rule(rung_k_dirs, rung_km1_dirs):
    """mean(Delta) / SE(Delta) of the seed-level mean paired fold-matched OOF ΔAUC
    (rung k vs rung k-1) -- DOCS/temporal-first-ablation.md 'The stopping rule',
    Tier 2's own definition. This is the statistic that governs keep/drop."""
    seed_means = []
    for exp_id_k, dir_k in rung_k_dirs.items():
        seed = exp_id_k.rsplit('seed', 1)[-1]
        matches = [d for eid, d in rung_km1_dirs.items() if eid.endswith(f'seed{seed}')]
        if not matches:
            continue
        auc_k, auc_km1 = per_fold_auc(dir_k), per_fold_auc(matches[0])
        common_folds = sorted(set(auc_k) & set(auc_km1))
        if not common_folds:
            continue
        seed_means.append(float(np.mean([auc_k[f] - auc_km1[f] for f in common_folds])))
    if len(seed_means) < 2:
        return {'mean': float('nan'), 'se': float('nan'), 'ratio': float('nan'),
                'n_seeds': len(seed_means), 'seed_means': seed_means, 'kept': None}
    mean = float(np.mean(seed_means))
    se = float(np.std(seed_means, ddof=1) / np.sqrt(len(seed_means)))
    return {'mean': mean, 'se': se, 'ratio': (mean / se) if se > 0 else float('nan'),
            'n_seeds': len(seed_means), 'seed_means': seed_means, 'kept': mean > se}


def pooled_stopping_rule(rung_k_dirs, rung_km1_dirs):
    """Secondary sanity statistic (2026-08-24 Tier-2 clarification,
    DOCS/temporal-first-ablation.md): each seed's pooled run_summary['oof']['oof_auc']
    differenced directly, WITHOUT fold pairing -- noisier than stopping_rule() above,
    never the keep/drop statistic on its own. Reported alongside it; the two must be
    printed together whenever they disagree in sign or keep/drop, per that
    clarification, rather than reporting only the number that supports a conclusion."""
    def pooled_auc(run_dir):
        p = run_dir / 'run_summary.json'
        if not p.is_file():
            return None
        s = json.loads(p.read_text())
        return s.get('oof', {}).get('oof_auc')

    seed_deltas = []
    for exp_id_k, dir_k in rung_k_dirs.items():
        seed = exp_id_k.rsplit('seed', 1)[-1]
        matches = [d for eid, d in rung_km1_dirs.items() if eid.endswith(f'seed{seed}')]
        if not matches:
            continue
        auc_k, auc_km1 = pooled_auc(dir_k), pooled_auc(matches[0])
        if auc_k is None or auc_km1 is None:
            continue
        seed_deltas.append(auc_k - auc_km1)
    if len(seed_deltas) < 2:
        return {'mean': float('nan'), 'se': float('nan'), 'ratio': float('nan'),
                'n_seeds': len(seed_deltas), 'seed_deltas': seed_deltas, 'kept': None}
    mean = float(np.mean(seed_deltas))
    se = float(np.std(seed_deltas, ddof=1) / np.sqrt(len(seed_deltas)))
    return {'mean': mean, 'se': se, 'ratio': (mean / se) if se > 0 else float('nan'),
            'n_seeds': len(seed_deltas), 'seed_deltas': seed_deltas, 'kept': mean > se}


def report_contrast(label, a, b):
    """Print both Tier-2 statistics for rung b vs rung a; flag disagreement (2026-08-24
    Tier-2 clarification -- never report only the statistic that supports a conclusion)."""
    fm = stopping_rule(RUNG_RUN_DIRS.get(b, {}), RUNG_RUN_DIRS.get(a, {}))
    pl = pooled_stopping_rule(RUNG_RUN_DIRS.get(b, {}), RUNG_RUN_DIRS.get(a, {}))
    fm_verdict = ('worth carrying forward' if fm['kept'] else
                  'undetectable at this sample size' if fm['kept'] is not None else
                  'not enough seeds resolved yet')
    print(f"  {label} ({b} vs {a})")
    print(f"    fold-matched: mean(D)={fm['mean']:.5f} SE(D)={fm['se']:.5f} "
          f"ratio={fm['ratio']:.2f} n_seeds={fm['n_seeds']} -> {fm_verdict}")
    print(f"    pooled:       mean(D)={pl['mean']:.5f} SE(D)={pl['se']:.5f} "
          f"ratio={pl['ratio']:.2f} n_seeds={pl['n_seeds']}")
    if fm['kept'] is not None and pl['kept'] is not None and fm['kept'] != pl['kept']:
        print(f"    *** DISAGREEMENT: fold-matched kept={fm['kept']} but pooled kept={pl['kept']} -- "
              f"apply the Tier-2 one-SE tie-breaker (next cell) rather than picking one reading. ***")
    return fm, pl


print('Chain-adjacent stopping-rule decisions (OOF):')
for k in range(1, len(RUNG_CHAIN)):
    a, b = RUNG_CHAIN[k - 1], RUNG_CHAIN[k]
    report_contrast(f'{b} vs {a}', a, b)

print()
print('Sensitivity contrasts (not part of RUNG_CHAIN, never selected on):')
for label, (a, b) in SENSITIVITY_CONTRASTS.items():
    report_contrast(label, a, b)

print()
print('Headline contrasts (isolate the flip itself):')
for label, (a, b) in HEADLINE_CONTRASTS.items():
    report_contrast(label, a, b)


## Tier 2 tie-breaker -- one-SE simplicity rule

Pre-registered in `DOCS/temporal-first-ablation.md` Tier 2, before any of the numbers
below existed: "among kept arms, prefer the simplest configuration within one SE of the
best." This cell computes the selection, it does not assert it -- the S1-over-S1b
decision (and any future tie) falls out of this rule mechanically, not by hand.

In [ ]:
def one_se_tie_breaker(candidates):
    """candidates: {name: (pooled_oof_auc_mean, pooled_oof_auc_se, complexity_rank)}
    where a LOWER complexity_rank means simpler (fewer dependencies / knobs active).
    Returns the name of the simplest candidate within one SE of the best mean --
    Tier 2's pre-registered modified one-standard-error rule."""
    best_name = max(candidates, key=lambda n: candidates[n][0])
    best_mean, best_se, _ = candidates[best_name]
    within_one_se = [
        n for n, (mean, se, _rank) in candidates.items()
        if best_mean - mean <= max(best_se, se)
    ]
    return min(within_one_se, key=lambda n: candidates[n][2])


def _pooled_oof_auc_mean_se(rung_name):
    rows = RUNG_TABLES.get(rung_name)
    if rows is None or rows.empty or 'oof_auc' not in rows.columns:
        return None
    vals = rows['oof_auc'].to_numpy()
    if len(vals) < 2:
        return None
    return float(vals.mean()), float(vals.std(ddof=1) / np.sqrt(len(vals)))


# S1 vs S1b: S1 is simpler (no SSL node-LSTM checkpoint dependency) -> lower rank.
_s1 = _pooled_oof_auc_mean_se('S1_flip')
_s1b = _pooled_oof_auc_mean_se('S1b_ssl')
if _s1 is not None and _s1b is not None:
    candidates = {
        'S1_flip': (_s1[0], _s1[1], 0),
        'S1b_ssl': (_s1b[0], _s1b[1], 1),
    }
    selected = one_se_tie_breaker(candidates)
    print('Tier-2 one-SE tie-breaker over {S1_flip, S1b_ssl}:')
    for n, (mean, se, rank) in candidates.items():
        print(f"  {n}: pooled OOF AUC mean={mean:.4f} SE={se:.4f} complexity_rank={rank}")
    print(f"  -> selected: {selected} (primary arm; the other is retained as a "
          f"documented sensitivity arm, secondary at Tier 4)")
else:
    print('S1 and/or S1b OOF rows not available yet -- tie-breaker not computed.')


## SENS-restricted comparison (PLAN.md section E)

SENS's 140 subjects (`min_visits=3`) are a strict subset of S1's 248. Two separable statements (PLAN.md section E) -- report both, never SENS's pooled OOF AUC next to S1's full-pool number as if fold-matched:

1. **Sequence-length evidence**: restrict S1's own OOF predictions to the same 140 subjects -- does the >=3-visit subgroup score higher for the model that saw the full 248-subject pool?
2. **Sample-size result**: compare that restricted S1 read to SENS's own OOF AUC (trained on only the 140-subject pool) -- does training on fewer, longer-sequence subjects help or hurt?


In [ ]:
def restrict_oof_auc(rung_name, subject_ids):
    """Per-seed AUC of rung_name's own OOF predictions, restricted to subject_ids."""
    aucs = {}
    for exp_id, run_dir in RUNG_RUN_DIRS.get(rung_name, {}).items():
        path = run_dir / 'oof_predictions.csv'
        if not path.is_file():
            continue
        df = pd.read_csv(path)
        sub = df[df['subject_id'].isin(subject_ids)]
        if sub['label'].nunique() < 2:
            continue
        seed = int(exp_id.rsplit('seed', 1)[-1])
        aucs[seed] = roc_auc_score(sub['label'], sub['prob'])
    return aucs


_sens_dirs = RUNG_RUN_DIRS.get('SENS', {})
if _sens_dirs:
    _any_sens_run = next(iter(_sens_dirs.values()))
    SENS_SUBJECT_IDS = set(pd.read_csv(_any_sens_run / 'oof_predictions.csv')['subject_id'].unique())
    print(f'SENS subject pool: {len(SENS_SUBJECT_IDS)} subjects (>= 3 visits)')

    _s1_restricted = restrict_oof_auc('S1_flip', SENS_SUBJECT_IDS)
    _sens_own = restrict_oof_auc('SENS', SENS_SUBJECT_IDS)
    _rows = [{'seed': s, 'S1_restricted_to_sens_pool': _s1_restricted.get(s),
              'SENS_trained_on_sens_pool': _sens_own.get(s)}
             for s in sorted(set(_s1_restricted) | set(_sens_own))]
    SENS_RESTRICTED_TABLE = pd.DataFrame(_rows).set_index('seed')
    print(SENS_RESTRICTED_TABLE)

    _s1_mean = SENS_RESTRICTED_TABLE['S1_restricted_to_sens_pool'].mean()
    _sens_mean = SENS_RESTRICTED_TABLE['SENS_trained_on_sens_pool'].mean()
    _s1_full_mean = RUNG_TABLES['S1_flip']['oof_auc'].mean() if not RUNG_TABLES['S1_flip'].empty else float('nan')
    print()
    print(f'1. Sequence-length evidence: S1 restricted to the >=3-visit pool = {_s1_mean:.4f} '
          f'vs S1 on the full 248-subject pool = {_s1_full_mean:.4f} -- the >=3-visit '
          f'subgroup is easier for the same trained model (positive signal).')
    print(f'2. Sample-size result: training only on the >=3-visit pool = {_sens_mean:.4f}, '
          f'{"worse" if _sens_mean < _s1_mean else "better"} than scoring the identical '
          f'subjects with the full-pool-trained S1 ({_s1_mean:.4f}) -- shrinking the training '
          f'pool 248->140 {"costs more than" if _sens_mean < _s1_mean else "is outweighed by"} '
          f'the longer-sequence gain.')
    print('Both read as "too small to decide on its own" per PLAN.md section E; SENS\'s pooled '
          'OOF AUC in RUNG_SUMMARY_TABLE is never a like-for-like row against S1\'s full-pool number.')
else:
    print('SENS: no runs yet -- restricted comparison is a no-op.')


## Tier 3 — robustness vetoes

Thresholds are fixed in `DOCS/temporal-first-ablation.md`'s addendum and never adjusted after seeing a result.

In [ ]:
VETO_THRESHOLDS = {
    'threshold_sd': 0.15,      # SD of best_threshold across 5 folds x 4 seeds
    'ece_oof_cal': 0.10,       # temperature-scaled OOF ECE
    'scan_count_spearman': 0.3,  # |r| of prob vs n_scans, within-stable
}


def veto_row(name, run_dirs):
    df = rung_oof_rows(run_dirs)
    if df.empty:
        return {'rung': name, 'n_seeds': 0}
    thresholds = []
    for run_dir in run_dirs.values():
        summary = load_summary(run_dir)
        if summary is None:
            continue
        thresholds.extend(summary.get('cv_results', {}).get('best_threshold', []))
    threshold_sd = float(np.std(thresholds, ddof=1)) if len(thresholds) > 1 else float('nan')

    row = {
        'rung': name,
        'threshold_sd': threshold_sd,
        'threshold_sd_veto': threshold_sd > VETO_THRESHOLDS['threshold_sd'],
        'ece_oof_cal': df['ece_oof_cal'].mean() if 'ece_oof_cal' in df else float('nan'),
    }
    row['ece_veto'] = (row['ece_oof_cal'] > VETO_THRESHOLDS['ece_oof_cal']
                        if pd.notna(row['ece_oof_cal']) else None)
    if 'oof_prob_nscans_spearman_non_converter' in df.columns:
        r = df['oof_prob_nscans_spearman_non_converter'].mean()
        row['scan_count_spearman_non_converter'] = r
        row['scan_count_veto'] = abs(r) > VETO_THRESHOLDS['scan_count_spearman'] if pd.notna(r) else None
    if 'cohort_probe_auc' in df.columns:
        cpa = df['cohort_probe_auc'].mean()
        row['cohort_probe_auc'] = cpa
        row['cohort_probe_escalation'] = cpa > 0.75 if pd.notna(cpa) else None
    demo_auc_by_cohort = {c: DEMO_FLOOR[c].mean() for c in DEMO_FLOOR.columns
                           if c.startswith('oof_auc_') and c != 'oof_auc'} if not DEMO_FLOOR.empty else {}
    for c, demo_auc in demo_auc_by_cohort.items():
        if c in df.columns:
            row[f'{c}_vs_demo_floor'] = df[c].mean() - demo_auc
            row[f'{c}_collapse_veto'] = df[c].mean() < demo_auc
    return row


VETO_TABLE = pd.DataFrame([veto_row(name, dirs) for name, dirs in RUNG_RUN_DIRS.items()]).set_index('rung')
VETO_TABLE


## Scan-count-shortcut mechanism (kept arms)

`common.visit_confound.within_subject_prob_slopes` needs a reloaded model + the `per_visit_probs` hook, not just the OOF frame -- run on the CV pool (never the test set) for a specific kept arm's best-fold checkpoint by setting `MECHANISM_CHECK_EXP_ID` below.

In [ ]:
MECHANISM_CHECK_EXP_ID = None  # e.g. 'tfgn-s1b-ssl-pooled-seed42' -- set to run this cell

if MECHANISM_CHECK_EXP_ID:
    from adapters import get_adapter
    from common.visit_confound import within_subject_prob_slopes
    from DATA.DELCODE.src.splitting.load_splits import splits_dir

    run_dir = resolve_source_run(MECHANISM_CHECK_EXP_ID, classifier_root=model_root)
    summary = load_summary(run_dir)
    if summary is None:
        raise FileNotFoundError(f'{MECHANISM_CHECK_EXP_ID}: no run_summary.json yet.')
    gaae_hp_path = model_root / 'configs' / 'gaae_delcode_whole_brain.json'
    gaae_hp = json.loads(gaae_hp_path.read_text()) if gaae_hp_path.is_file() else {}

    pooled_splits = repo_root / 'DATA' / 'POOLED_ADNI_DELCODE' / 'SPLITS' / 'downstream'
    cv_pool_df = pd.concat([pd.read_csv(pooled_splits / 'train.csv'),
                            pd.read_csv(pooled_splits / 'val.csv')], ignore_index=True)

    adapter_key = str(summary.get('model_config', {}).get('model_type', '')).lower()
    # model_type is the class name; map every family, not only the TFGN one --
    # the matched-window reference arm (PLAN.md section F) is a GELSTM, and an
    # unmapped key fails only at get_adapter(), after the frame is already built.
    adapter_key = {'tfgnclassifier': 'tfgn', 'logregdriftadapter': 'logregdrift',
                   'gelstmclassifier': 'gelstm',
                   'braintokengtclassifier': 'braintokengt'}.get(adapter_key, adapter_key)
    adapter = get_adapter(adapter_key)(
        gaae_ckpt_path=summary.get('gaae_checkpoint') or '', gaae_hp=gaae_hp,
        train_config=summary['training_config'],
        data_root=str(repo_root / 'DATA/POOLED_ADNI_DELCODE/__fc_wholebrain_sch200_flat__/matrices'),
        cohorts_csv=None, device='cpu', rng=None,
    )
    state = adapter.load_state(run_dir)
    cv_bundle = adapter.prepare_data(cv_pool_df)
    slope_df, slope_stats = within_subject_prob_slopes(cv_bundle, adapter.per_visit_probs, state, device='cpu')
    print(f'Within-subject slope of P(converter) vs visit index -- {MECHANISM_CHECK_EXP_ID} (CV pool, not test):')
    for grp, s in slope_stats.items():
        print(f"  {grp:14s} median_slope={s['median_slope']:.4f}  frac_negative={s['frac_negative']}  n={s['n']}")
else:
    print('MECHANISM_CHECK_EXP_ID not set -- skipping (set it to a kept arm\'s seed id to run).')


## Gate-map validation (S2, S5) -- section 0.1d, DOCS/temporal-first-ablation.md

S2 (learned gate) was dropped by the stopping rule and S5 (dual-score) carries no learned temporal axis, so this runs on the pre-registered (2026-08-24 addendum) pair: **`s_topo`** (S5's `dual_scores.npy`, the primary, kept interpretability arm) against **`d_tilde`** (the model-free rank-sigmoid drift anchor, `model/TFGN/dataset.py::compute_drift_anchor`, computed offline at zero GPU cost). S2's `gate_scores.npy` is reported alongside as a **supporting panel** from a rejected arm, never the primary axis (PLAN.md section C).

**Two further documented deviations, on top of the cross-fold-to-cross-seed reduction already recorded in the doc (PLAN.md section C, artifact limitation):**

1. **DMN only, not DMN/hippocampal.** The whole-brain atlas TFGN actually consumes (`__fc_wholebrain_sch200_flat__`, Schaefer-200 cortical parcellation) carries no hippocampal or other subcortical ROI -- that requires the separate `__fc_dmn-hippo_sch200-tian2_flat__` data product, which no TFGN rung reads. The overlap check below is restricted to the Yeo-7 `Default` (DMN) network, the 46/200 ROIs the atlas actually contains.
2. **Permutation-null design.** The pre-registration's "1000 label permutations" is ambiguous for a spatial overlap statistic (DMN membership is an anatomical label, not a subject label). **Decision: a network-label spin test** -- 1000 permutations reassigning which 46 of the 200 nodes are labelled DMN (uniform without replacement, preserving the true DMN count), each time recomputing the overlap with the *fixed, observed* top-`round(gate_rho*200)`-score node set; the observed overlap's percentile against this null is the reported statistic. This tests "is the gate's node ranking enriched for the DMN label more than a random 46-node subset would be", the operative question given the atlas actually available.


In [ ]:
from scipy.stats import spearmanr
from adapters import get_adapter

_ATLAS_PATH = repo_root / 'DASHBOARD' / 'app' / 'static' / 'data' / 'schaefer_200_coords.json'
_atlas = json.loads(_ATLAS_PATH.read_text())
_rois = sorted(_atlas['rois'], key=lambda r: r['index'])
assert [r['index'] for r in _rois] == list(range(len(_rois)))
IS_DMN = np.array([r['network'] == 'Default' for r in _rois])
N_ROIS = len(_rois)
GATE_RHO_MAP = 0.15  # matches every TFGN rung's gate_rho (resolved_config.json, all seeds)
TOP_K = round(GATE_RHO_MAP * N_ROIS)
N_PERM = 1000
_perm_rng = np.random.default_rng(20260825)

_pooled_dir = repo_root / 'DATA' / 'POOLED_ADNI_DELCODE'
_pooled_splits = _pooled_dir / 'SPLITS' / 'downstream'
_cv_pool_df = pd.concat(
    [pd.read_csv(_pooled_splits / 'train.csv'), pd.read_csv(_pooled_splits / 'val.csv')],
    ignore_index=True,
)


def load_node_maps(rung_name, artifact_name):
    """seed -> dict(maps, cohort, sids, node_mean, training_config) for one rung's best-fold artifact."""
    out = {}
    for exp_id, run_dir in RUNG_RUN_DIRS.get(rung_name, {}).items():
        arr_path = run_dir / artifact_name
        if not arr_path.is_file():
            continue
        arr = np.load(arr_path)
        cohort = np.load(run_dir / 'cohort_tags.npy', allow_pickle=True)
        summary = load_summary(run_dir)
        oof = pd.read_csv(run_dir / 'oof_predictions.csv')
        fold_df = oof[oof['fold'] == summary['best_fold']].reset_index(drop=True)
        assert (fold_df['cohort'].values == cohort).all(), f'{exp_id}: cohort/order mismatch'
        seed = int(exp_id.rsplit('seed', 1)[-1])
        out[seed] = {'maps': arr, 'cohort': cohort, 'sids': fold_df['subject_id'].tolist(),
                     'node_mean': arr.mean(axis=0), 'training_config': summary['training_config']}
    return out


def drift_anchor_matrix(sids, training_config):
    """(len(sids), 200) d_tilde, zero GPU cost -- pure function of each subject's own X."""
    adapter = get_adapter('tfgn')(
        gaae_ckpt_path='', gaae_hp={}, train_config=training_config,
        data_root=str(_pooled_dir / '__fc_wholebrain_sch200_flat__' / 'matrices'),
        cohorts_csv=None, device='cpu', rng=None,
    )
    sub_df = _cv_pool_df[_cv_pool_df['subject_id'].isin(sids)].copy()
    bundle = adapter.prepare_data(sub_df)
    tfgn_items = adapter._prepare_tfgn_items(bundle.items)
    da_by_sid = {it.subject_id: it.drift_anchor.numpy() for it in tfgn_items}
    missing = [s for s in sids if s not in da_by_sid]
    if missing:
        raise ValueError(f'drift_anchor_matrix: missing subjects {missing}')
    return np.stack([da_by_sid[s] for s in sids], axis=0)


def dmn_overlap(node_mean_vec):
    top_idx = np.argsort(node_mean_vec)[::-1][:TOP_K]
    return int(IS_DMN[top_idx].sum())


def dmn_spin_test(node_mean_vec, n_perm=N_PERM, rng=_perm_rng):
    """Percentile of the observed DMN overlap against a null built by permuting which nodes
    carry the DMN label (n_dmn drawn without replacement from all 200), holding the observed
    top-TOP_K node set fixed -- see the markdown cell above for why this design, not a
    literal cortical-surface spin test (no spherical projection infra in this repo)."""
    observed = dmn_overlap(node_mean_vec)
    top_idx = np.argsort(node_mean_vec)[::-1][:TOP_K]
    is_top = np.zeros(N_ROIS, dtype=bool)
    is_top[top_idx] = True
    n_dmn = int(IS_DMN.sum())
    idx_all = np.arange(N_ROIS)
    null = np.array([int(is_top[rng.choice(idx_all, size=n_dmn, replace=False)].sum())
                      for _ in range(n_perm)])
    return {'observed_overlap': observed, 'k': TOP_K, 'n_dmn': n_dmn,
            'null_mean': float(null.mean()), 'null_sd': float(null.std()),
            'percentile': float((null <= observed).mean() * 100),
            'p_value_one_sided': float((null >= observed).mean())}


def cross_seed_spearman(node_means_by_seed):
    seeds = sorted(node_means_by_seed)
    if len(seeds) < 2:
        return {'n_seeds': len(seeds), 'mean_r': float('nan')}
    rs = [spearmanr(node_means_by_seed[seeds[i]], node_means_by_seed[seeds[j]])[0]
          for i in range(len(seeds)) for j in range(i + 1, len(seeds))]
    ref = seeds[0]
    vs_ref = {f'{s}_vs_{ref}': float(spearmanr(node_means_by_seed[ref], node_means_by_seed[s])[0])
              for s in seeds[1:]}
    return {'n_seeds': len(seeds), 'mean_r': float(np.mean(rs)), 'min_r': float(np.min(rs)),
            'max_r': float(np.max(rs)), 'vs_ref_seed': ref, 'vs_ref': vs_ref}


def per_cohort_validation(maps_by_seed, d_tilde_by_seed, cohort_name):
    node_means_s, node_means_d, overlaps_s, overlaps_d = {}, {}, [], []
    for seed, d in maps_by_seed.items():
        mask = d['cohort'] == cohort_name
        if not mask.any():
            continue
        nm_s, nm_d = d['maps'][mask].mean(axis=0), d_tilde_by_seed[seed][mask].mean(axis=0)
        node_means_s[seed], node_means_d[seed] = nm_s, nm_d
        overlaps_s.append(dmn_overlap(nm_s))
        overlaps_d.append(dmn_overlap(nm_d))
    return {
        'n_seeds': len(node_means_s),
        's_topo_dmn_overlap_mean': float(np.mean(overlaps_s)) if overlaps_s else None,
        'd_tilde_dmn_overlap_mean': float(np.mean(overlaps_d)) if overlaps_d else None,
        's_topo_cross_seed_spearman': cross_seed_spearman(node_means_s) if len(node_means_s) >= 2 else None,
        'd_tilde_cross_seed_spearman': cross_seed_spearman(node_means_d) if len(node_means_d) >= 2 else None,
    }


def quadrant_counts(s_vec, d_vec):
    s_med, d_med = np.median(s_vec), np.median(d_vec)
    r, p = spearmanr(s_vec, d_vec)
    return {'HH': int(((s_vec >= s_med) & (d_vec >= d_med)).sum()),
            'HL': int(((s_vec >= s_med) & (d_vec < d_med)).sum()),
            'LH': int(((s_vec < s_med) & (d_vec >= d_med)).sum()),
            'LL': int(((s_vec < s_med) & (d_vec < d_med)).sum()),
            'spearman_r': float(r), 'spearman_p': float(p)}


GATE_MAP_VALIDATION = {}
for _rung, _prefix_key, _artifact in [('S5_dualscore', 'dual_scores.npy', 'dual_scores.npy'),
                                       ('S2_gate', 'gate_scores.npy', 'gate_scores.npy')]:
    maps_by_seed = load_node_maps(_rung, _artifact)
    if not maps_by_seed:
        print(f'{_rung}: no {_artifact} artifacts yet -- gate-map validation is a no-op.')
        continue
    d_tilde_by_seed = {s: drift_anchor_matrix(d['sids'], d['training_config'])
                        for s, d in maps_by_seed.items()}
    s_node_means = {s: d['node_mean'] for s, d in maps_by_seed.items()}
    d_node_means = {s: d_tilde_by_seed[s].mean(axis=0) for s in maps_by_seed}
    s_avg, d_avg = np.mean(list(s_node_means.values()), axis=0), np.mean(list(d_node_means.values()), axis=0)

    result = {
        's_topo': {'dmn_spin_test': dmn_spin_test(s_avg), 'cross_seed_spearman': cross_seed_spearman(s_node_means)},
        'd_tilde': {'dmn_spin_test': dmn_spin_test(d_avg), 'cross_seed_spearman': cross_seed_spearman(d_node_means)},
        'quadrant_scatter': quadrant_counts(s_avg, d_avg),
        'per_cohort': {'adni': per_cohort_validation(maps_by_seed, d_tilde_by_seed, 'adni'),
                       'delcode': per_cohort_validation(maps_by_seed, d_tilde_by_seed, 'delcode')},
        '_s_avg': s_avg, '_d_avg': d_avg,
    }
    GATE_MAP_VALIDATION[_rung] = result

    label = 'PRIMARY (S5, kept interpretability arm)' if _rung == 'S5_dualscore' else 'SUPPORTING PANEL (S2, dropped by Tier 2)'
    print(f'=== {_rung} -- {label} ===')
    print(f"  s_topo DMN overlap: {result['s_topo']['dmn_spin_test']['observed_overlap']}/{TOP_K} "
          f"(null mean {result['s_topo']['dmn_spin_test']['null_mean']:.2f}, "
          f"percentile {result['s_topo']['dmn_spin_test']['percentile']:.1f}, "
          f"p={result['s_topo']['dmn_spin_test']['p_value_one_sided']:.3f})")
    print(f"  d_tilde DMN overlap: {result['d_tilde']['dmn_spin_test']['observed_overlap']}/{TOP_K} "
          f"(null mean {result['d_tilde']['dmn_spin_test']['null_mean']:.2f}, "
          f"percentile {result['d_tilde']['dmn_spin_test']['percentile']:.1f}, "
          f"p={result['d_tilde']['dmn_spin_test']['p_value_one_sided']:.3f})")
    print(f"  cross-seed Spearman -- s_topo: mean r={result['s_topo']['cross_seed_spearman']['mean_r']:.3f} "
          f"[{result['s_topo']['cross_seed_spearman']['min_r']:.3f}, {result['s_topo']['cross_seed_spearman']['max_r']:.3f}]  "
          f"d_tilde: mean r={result['d_tilde']['cross_seed_spearman']['mean_r']:.3f} "
          f"[{result['d_tilde']['cross_seed_spearman']['min_r']:.3f}, {result['d_tilde']['cross_seed_spearman']['max_r']:.3f}]")
    qs = result['quadrant_scatter']
    print(f"  quadrant scatter (s_topo vs d_tilde, cross-seed-averaged map): "
          f"HH={qs['HH']} HL={qs['HL']} LH={qs['LH']} LL={qs['LL']}  "
          f"Spearman r={qs['spearman_r']:.3f} (p={qs['spearman_p']:.2e})")
    for _cohort in ('adni', 'delcode'):
        pc = result['per_cohort'][_cohort]
        print(f"  [{_cohort}] n_seeds={pc['n_seeds']}  s_topo DMN overlap mean={pc['s_topo_dmn_overlap_mean']}  "
              f"d_tilde DMN overlap mean={pc['d_tilde_dmn_overlap_mean']}")
    print()

if 'S5_dualscore' in GATE_MAP_VALIDATION:
    _fig, _axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
    for _ax, _rung, _title in zip(
        _axes, ['S5_dualscore', 'S2_gate'],
        ['S5 s_topo (primary) vs d_tilde', 'S2 gate_scores (supporting, dropped) vs d_tilde'],
    ):
        if _rung not in GATE_MAP_VALIDATION:
            _ax.set_visible(False)
            continue
        _s, _d = GATE_MAP_VALIDATION[_rung]['_s_avg'], GATE_MAP_VALIDATION[_rung]['_d_avg']
        _ax.scatter(_d, _s, c=IS_DMN, cmap='coolwarm', s=18, alpha=0.8)
        _ax.axvline(np.median(_d), color='grey', lw=0.8, ls='--')
        _ax.axhline(np.median(_s), color='grey', lw=0.8, ls='--')
        _ax.set_xlabel('d_tilde (drift anchor, node mean)')
        _ax.set_ylabel('learned score (node mean)')
        _ax.set_title(_title)
    plt.tight_layout()
    plt.show()


## Table A / Table B (Matched-window SOTA comparison addendum, DOCS/flipped/PLAN.md)

Table A (T in [2,3], strict head-to-head against the SOTA competitor under its own input constraint) and Table B (the ladder as registered, full trajectory, the thesis contribution claim). Never presented as evidence for or against the flip on its own -- PLAN.md section F.


In [ ]:
# Guard (checklist item): the matched-window arms must share Block A's exact CV pool
# before Table A means anything -- PLAN.md "S0d reuse -- the guard, not just the assumption".
def _oof_subject_set(rung_name):
    ids = set()
    for run_dir in RUNG_RUN_DIRS.get(rung_name, {}).values():
        p = run_dir / 'oof_predictions.csv'
        if p.is_file():
            ids |= set(pd.read_csv(p)['subject_id'].unique())
    return ids


_block_a_pool = _oof_subject_set('S1_flip')
for _w3_rung in ['W3_gelstm_frozen', 'W3_gelstm_random', 'W3_tfgn_winner']:
    _pool = _oof_subject_set(_w3_rung)
    if _pool:
        assert _pool == _block_a_pool, (
            f'{_w3_rung} CV pool ({len(_pool)}) != Block A pool ({len(_block_a_pool)}) -- '
            'matched-window guard failed, PLAN.md "S0d reuse -- the guard, not just the assumption".'
        )
if _block_a_pool:
    print(f'Matched-window guard OK: W3 arms share the {len(_block_a_pool)}-subject Block A CV pool.')
else:
    print('Block A pool not resolved yet (S1_flip has no OOF rows) -- guard skipped.')


In [ ]:
TABLE_A_RUNGS = ['S0d_braintokengt', 'W3_gelstm_random', 'W3_gelstm_frozen', 'W3_tfgn_winner']
_table_a_rows = [r for r in TABLE_A_RUNGS if r in RUNG_SUMMARY_TABLE.index]
TABLE_A = RUNG_SUMMARY_TABLE.loc[_table_a_rows, ['n_seeds', 'oof_auc_mean', 'oof_auc_sd']] \
    if _table_a_rows else pd.DataFrame()
print('Table A -- matched window (T in [2,3]), strict head-to-head vs the SOTA competitor:')
print(TABLE_A)
print()
print('Fold-matched contrasts within Table A:')
if 'W3_gelstm_frozen' in RUNG_RUN_DIRS and 'W3_tfgn_winner' in RUNG_RUN_DIRS:
    report_contrast('W3-TFGN vs W3-GELSTM-frozen', 'W3_gelstm_frozen', 'W3_tfgn_winner')
if 'S0d_braintokengt' in RUNG_RUN_DIRS and 'W3_tfgn_winner' in RUNG_RUN_DIRS:
    report_contrast('W3-TFGN vs BrainTokenGT (S0d)', 'S0d_braintokengt', 'W3_tfgn_winner')
print()
print('Table A framing (PLAN.md section F): under the competitor\'s short-window constraint '
      'the temporal-first flip loses its advantage over spatial-first (W3-TFGN vs '
      'W3-GELSTM-frozen above), while still beating the SOTA competitor itself (vs '
      'BrainTokenGT above). Never read Table A as evidence for or against the flip -- Table B '
      'carries that claim; Table A shows the flip\'s gain is a long-sequence gain that a short '
      'window throws away, which is the confirmation of Table B\'s claim, not a contradiction.')


In [ ]:
TABLE_B_RUNGS = ['S0_demo', 'S0a_logreg_drift', 'S0b_gelstm_frozen', 'S0c_gelstm_random',
                  'S1_flip', 'S1b_ssl', 'S1c_recon_random', 'S2_gate', 'S3_fusion',
                  'S4_attnpool', 'S5_dualscore', 'SENS']
_table_b_rows = [r for r in TABLE_B_RUNGS if r in RUNG_SUMMARY_TABLE.index]
TABLE_B = RUNG_SUMMARY_TABLE.loc[_table_b_rows] if _table_b_rows else pd.DataFrame()
print('Table B -- the ladder as registered (full trajectory, T>=2), the thesis contribution claim:')
TABLE_B


## Tier 4 — frozen reads (in-domain test + OASIS-3, exactly once)

**Gated.** Nothing below executes unless `RUN_FROZEN_READ = True` and `FROZEN_WINNER_ID` names the frozen winning arm's id prefix (no seed suffix -- all 4 seeds are read). Uses `common.frozen_read.score_frozen_split` -- reloads each seed's saved checkpoint, scores at its own OOF-derived threshold, records via the same `record_test_metrics` / `record_external_metrics` every non-deferred run already uses.

In [ ]:
def run_frozen_reads(id_prefix, label):
    """One Tier-4 frozen-read pass (in-domain test + OASIS-3, all 4 seeds) for a
    single arm. `label` is 'PRIMARY' or 'SECONDARY (sensitivity arm, not primary)' --
    printed on every line so a reader of run.log / notebook output can never mistake
    a secondary sensitivity read for the primary estimate."""
    from common.frozen_read import score_frozen_split
    from DATA.DELCODE.src.splitting.load_splits import splits_dir  # noqa: F401

    gaae_hp_path = model_root / 'configs' / 'gaae_delcode_whole_brain.json'
    gaae_hp = json.loads(gaae_hp_path.read_text()) if gaae_hp_path.is_file() else {}
    pooled_dir = repo_root / 'DATA' / 'POOLED_ADNI_DELCODE'
    in_domain_test_df = pd.read_csv(pooled_dir / 'SPLITS' / 'downstream' / 'test.csv')
    oasis_splits = repo_root / 'DATA' / 'OASIS3' / '__metadata__' / 'SPLITS' / 'downstream'
    oasis_test_df = pd.concat(
        [pd.read_csv(oasis_splits / f'{s}.csv') for s in ('train', 'val', 'test')], ignore_index=True,
    )
    oasis_test_df['cohort'] = 'oasis3'

    results = {}
    for seed in SEEDS:
        exp_id = f'{id_prefix}-seed{seed}'
        run_dir = resolve_source_run(exp_id, classifier_root=model_root)
        summary = load_summary(run_dir)
        if summary is None:
            raise FileNotFoundError(f'{exp_id}: no run_summary.json yet -- not ready for a frozen read.')
        adapter_key = str(summary.get('model_config', {}).get('model_type', '')).lower()
        # model_type is the class name; map every family, not only the TFGN one --
        # the matched-window reference arm (PLAN.md section F) is a GELSTM, and an
        # unmapped key fails only at get_adapter(), after the frame is already built.
        adapter_key = {'tfgnclassifier': 'tfgn', 'logregdriftadapter': 'logregdrift',
                       'gelstmclassifier': 'gelstm',
                       'braintokengtclassifier': 'braintokengt'}.get(adapter_key, adapter_key)
        common_kwargs = dict(
            adapter_key=adapter_key,
            data_root=str(pooled_dir / '__fc_wholebrain_sch200_flat__' / 'matrices'),
            cohorts_csv=None, gaae_ckpt_path=summary.get('gaae_checkpoint') or '',
            gaae_hp=gaae_hp, device='cpu',
        )
        test_metrics = score_frozen_split(run_dir, in_domain_test_df, record_as='test', **common_kwargs)
        ext_metrics = score_frozen_split(run_dir, oasis_test_df, record_as='external', cohort='oasis3', **common_kwargs)
        results[exp_id] = {'test_auc': test_metrics['auc'], 'ext_oasis3_auc': ext_metrics['auc']}
        print(f'[{label}] {exp_id}: test_auc={test_metrics["auc"]:.4f}  ext_oasis3_auc={ext_metrics["auc"]:.4f}')

    frozen_df = pd.DataFrame(results).T
    print()
    print(f'[{label}] Frozen reads across seeds ({id_prefix}):')
    print(frozen_df)
    print()
    print(f"[{label}] In-domain test AUC: {frozen_df['test_auc'].mean():.4f} +/- {frozen_df['test_auc'].std():.4f}")
    print(f"[{label}] OASIS-3 AUC:        {frozen_df['ext_oasis3_auc'].mean():.4f} +/- {frozen_df['ext_oasis3_auc'].std():.4f}")

    winner_oof = RUNG_TABLES.get(id_prefix)
    if winner_oof is None:
        for name, prefix in RUNG_PREFIXES.items():
            if prefix == id_prefix:
                winner_oof = RUNG_TABLES.get(name)
    if winner_oof is not None and not winner_oof.empty:
        se_oof = winner_oof['oof_auc'].std(ddof=1) / np.sqrt(len(winner_oof))
        se_test = frozen_df['test_auc'].std(ddof=1) / np.sqrt(len(frozen_df)) if len(frozen_df) > 1 else float('nan')
        half_width = 1.96 * np.sqrt(se_oof ** 2 + se_test ** 2)
        lo, hi = winner_oof['oof_auc'].mean() - half_width, winner_oof['oof_auc'].mean() + half_width
        consistent = lo <= frozen_df['test_auc'].mean() <= hi
        print()
        print(f'[{label}] Transport check: OOF={winner_oof["oof_auc"].mean():.4f}  '
              f'95% prediction interval=[{lo:.4f}, {hi:.4f}]  '
              f'test={frozen_df["test_auc"].mean():.4f}  '
              f'-> {"consistent" if consistent else "inconsistent"} with CV->test transport.')
        print(f'[{label}] Winner\'s-curse statement: the OOF AUC above is expected to be optimistic '
              '(selected as the best of the ladder); the frozen test read above is the '
              'unbiased estimate. Report both, not the OOF number alone, as the headline.')
    return frozen_df


if not RUN_FROZEN_READ:
    print('RUN_FROZEN_READ=False -- Tier 4 skipped (the ladder is not frozen yet, or '
          'this is a routine re-run of sections 1-6). Flip both flags above once ready.')
elif not FROZEN_WINNER_ID:
    raise ValueError('RUN_FROZEN_READ=True requires FROZEN_WINNER_ID (an id prefix).')
else:
    FROZEN_RESULTS = run_frozen_reads(FROZEN_WINNER_ID, 'PRIMARY')
    # SECONDARY_SENSITIVITY_ID: a single id prefix or a list of them, each read in
    # this same one-shot pass and reported side by side -- never substituted for the
    # primary. S5's label is specialised so its number reads as the interpretability
    # layer's estimate (PLAN.md section A: kept regardless of AUC), not a rival to S1.
    _SECONDARY_LABELS = {
        'tfgn-s5-dualscore-pooled': 'SECONDARY (interpretability layer, kept regardless of AUC -- not a competing endpoint)',
    }
    SECONDARY_RESULTS = {}
    _secondary_ids = SECONDARY_SENSITIVITY_ID if SECONDARY_SENSITIVITY_ID else []
    if isinstance(_secondary_ids, str):
        _secondary_ids = [_secondary_ids]
    for _sec_id in _secondary_ids:
        print()
        print('=' * 70)
        _label = _SECONDARY_LABELS.get(_sec_id, 'SECONDARY (sensitivity arm, not primary)')
        SECONDARY_RESULTS[_sec_id] = run_frozen_reads(_sec_id, _label)


## Guard check — sections 1-6 touched no test/external metric

In [ ]:
_forbidden = {'TEST_METRICS', 'EXTERNAL_METRICS', 'test_df', 'in_domain_test_df', 'oasis_test_df'}
_touched = _forbidden & set(dir())
if RUN_FROZEN_READ:
    print(f'RUN_FROZEN_READ=True -- Tier 4 ran by design; test/external names present: {_touched or "(pandas frames only, as expected)"}.')
else:
    _unexpected = _touched - {'test_df'}  # 'test_df' would only exist if RUN_FROZEN_READ ran
    assert not _unexpected, f'Sections 1-6 touched test/external state unexpectedly: {_unexpected}'
    print('OK -- no test/external metric was read (RUN_FROZEN_READ=False).')
